# 🚁 Drone Detection — YOLOv11 Training Pipeline
**Capstone Project | Jetson Nano Orin Deployment**

This notebook covers:
1. Environment setup
2. Dataset download (Roboflow open-source drone dataset)
3. Dataset verification
4. YOLOv11 training
5. Evaluation & metrics
6. Model export (PyTorch `.pt` → ready for TensorRT on Jetson)

> **Runtime:** Make sure you're on a GPU runtime.
> Go to `Runtime → Change runtime type → A100 GPU`

## ✅ Step 0 — Verify GPU

In [ ]:
import torch

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected — switch runtime to T4 GPU before training')

## 📦 Step 1 — Install Dependencies

In [ ]:
# Install Ultralytics (includes YOLOv11) and Roboflow SDK
!pip install ultralytics roboflow --quiet

# Fix Colab's recent polars dependency conflict causing plotting errors
!pip uninstall -y cudf-polars-cu12
!pip install -U polars seaborn --quiet

# Verify installation
import ultralytics
ultralytics.checks()  # prints environment info, CUDA status, YOLOv11 version


## 📂 Step 2 — Dataset Setup

**Option A** — Download from Roboflow Universe

**Option B** — Upload your own dataset

Run only ONE of the two options below.

### Option A — Roboflow Drone Dataset

In [ ]:
# ---------------------------------------------------------------
# OPTION A: Download a public drone dataset from Roboflow Universe
# ---------------------------------------------------------------
# Steps to get your own API key (free):
#   1. Go to https://universe.roboflow.com
#   2. Sign up / log in
#   3. Click your profile → Settings → Copy API key
#   4. Paste it below
#
# The dataset used here:
#   https://universe.roboflow.com/senior-projectdrone-dataset/capstone-drone-detection-dataset  << Access dataset from here!!
# ---------------------------------------------------------------

from roboflow import Roboflow

ROBOFLOW_API_KEY = "***************"  # ← paste your key here

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Drone detection dataset (The following information can be obtained from Roboflow after forking the dataset and exporting it)
project = rf.workspace("     ").project("     ")
version  = project.version(1)
dataset  = version.download("yolov11")

DATASET_YAML = dataset.location + "/data.yaml"
print("Dataset path:", dataset.location)
print("data.yaml path:", DATASET_YAML)

### Option B — Upload Your Own Dataset

In [ ]:
# ---------------------------------------------------------------
# OPTION B: Upload your dataset zip from local machine
# ---------------------------------------------------------------
# Your zip should have this structure:
#   dataset.zip
#   ├── train/images/ & train/labels/
#   ├── val/images/   & val/labels/
#   ├── test/images/  & test/labels/
#   └── data.yaml
# ---------------------------------------------------------------

from google.colab import files
import zipfile, os

print("Upload your dataset zip file:")
uploaded = files.upload()  # opens file picker

zip_name = list(uploaded.keys())[0]
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(extract_path)

print("Extracted to:", extract_path)
!find /content/dataset -maxdepth 3 -type d  # show folder structure

DATASET_YAML = "/content/dataset/data.yaml"  # adjust if needed

## 🔍 Step 3 — Dataset Verification

### 3.1 Dataset Size and Split Counts

First, we count the total number of images across our training, validation, and test splits. This ensures the dataset structure is recognized properly and gives us a baseline size for our final reports.

In [ ]:
# ── Count Dataset (used for reporting at the end of the file)
from pathlib import Path

dataset_root = Path(DATASET_YAML).parent

# Count images per split
split_counts = {}
total_images = 0

for split in ['train', 'valid', 'val', 'test']:
    img_dir = dataset_root / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
        # normalize split name for display
        split_name = 'val' if split == 'valid' else split
        split_counts[split_name] = count
        total_images += count

print("Dataset Size:")
for split, count in split_counts.items():
    print(f"  {split:8s}: {count} images")
print(f"  {'total':8s}: {total_images} images")

### 3.2 Annotation Verification and Class Balance

It is crucial to verify that every image has a corresponding label file. This step checks for missing annotations and analyzes the class distribution to warn us of any severe class imbalances that might negatively impact model training.

In [ ]:
# ── Verify dataset is annotated
import os
import yaml
from pathlib import Path

# ── Load and display dataset.yaml ──────────────────────────────
with open(DATASET_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

print("=" * 50)
print("DATASET CONFIG")
print("=" * 50)
for k, v in data_cfg.items():
    print(f"  {k}: {v}")

# ── Count images and labels per split ──────────────────────────
print("\n" + "=" * 50)
print("SPLIT SUMMARY")
print("=" * 50)

dataset_root = Path(DATASET_YAML).parent

for split in ['train', 'valid', 'test']:
    img_dir = dataset_root / split / 'images'
    lbl_dir = dataset_root / split / 'labels'

    if not img_dir.exists():
        # Some datasets use 'val' instead of 'valid'
        img_dir = dataset_root / split.replace('valid', 'val') / 'images'
        lbl_dir = dataset_root / split.replace('valid', 'val') / 'labels'

    if img_dir.exists():
        imgs = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
        lbls = list(lbl_dir.glob('*.txt')) if lbl_dir.exists() else []
        print(f"  {split:8s} → {len(imgs):4d} images | {len(lbls):4d} labels")

# ── Check for mismatched image/label pairs ──────────────────────
print("\n" + "=" * 50)
print("ANNOTATION CHECK")
print("=" * 50)

issues = []
for split in ['train', 'valid', 'val', 'test']:
    img_dir = dataset_root / split / 'images'
    lbl_dir = dataset_root / split / 'labels'
    if not img_dir.exists():
        continue
    for img_path in img_dir.glob('*.*'):
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            issues.append(f"Missing label: {img_path.name}")

if issues:
    print(f"  ⚠️  {len(issues)} images missing labels:")
    for i in issues[:10]:  # show first 10
        print(f"    {i}")
else:
    print("  ✅ All images have matching label files")

# ── Class distribution ──────────────────────────────────────────
print("\n" + "=" * 50)
print("CLASS DISTRIBUTION (train set)")
print("=" * 50)

from collections import Counter

class_counts = Counter()
train_lbl_dir = dataset_root / 'train' / 'labels'
if train_lbl_dir.exists():
    for lbl_file in train_lbl_dir.glob('*.txt'):
        with open(lbl_file) as f:
            for line in f:
                if line.strip():
                    class_id = int(line.split()[0])
                    class_counts[class_id] += 1

classes = data_cfg.get('names', {})
for cls_id, count in sorted(class_counts.items()):
    name = classes[cls_id] if isinstance(classes, list) else classes.get(cls_id, str(cls_id))
    print(f"  Class {cls_id} ({name}): {count} annotations")

if len(class_counts) > 1:
    counts = list(class_counts.values())
    ratio = max(counts) / min(counts)
    if ratio > 5:
        print(f"\n  ⚠️  Class imbalance detected (ratio {ratio:.1f}x) — consider augmentation")
    else:
        print(f"\n  ✅ Class balance looks good (ratio {ratio:.1f}x)")

### 3.3 Visualizing Sample Annotations

Finally, we visually inspect a few random training images with their bounding boxes overlaid. This serves as a sanity check to confirm that the bounding box coordinates are correctly formatted and aligned with the objects in the images.

In [ ]:
# ── Visualize sample annotations ───────────────────────────────
import cv2
import random
import matplotlib.pyplot as plt
import numpy as np

COLORS = [(0, 255, 0), (255, 100, 0), (0, 100, 255)]  # per class

def draw_yolo_boxes(img_path, lbl_path, class_names):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = int((xc - bw/2) * w)
                y1 = int((yc - bh/2) * h)
                x2 = int((xc + bw/2) * w)
                y2 = int((yc + bh/2) * h)
                color = COLORS[cls % len(COLORS)]
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                label = class_names[cls] if cls < len(class_names) else str(cls)
                cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
    return img

# Sample 6 random training images
train_img_dir = dataset_root / 'train' / 'images'
train_lbl_dir = dataset_root / 'train' / 'labels'
class_names = data_cfg.get('names', ['drone', 'bird'])
if isinstance(class_names, dict):
    class_names = [class_names[i] for i in sorted(class_names)]

img_files = list(train_img_dir.glob('*.jpg')) + list(train_img_dir.glob('*.png'))
samples = random.sample(img_files, min(6, len(img_files)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sample Training Annotations', fontsize=14, fontweight='bold')

for ax, img_path in zip(axes.flat, samples):
    lbl_path = train_lbl_dir / (img_path.stem + '.txt')
    img = draw_yolo_boxes(img_path, lbl_path, class_names)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_annotations.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: sample_annotations.png")

## 🏋️ Step 4 — Training


### 4.1 Adding the P2 Head

Standard YOLO architectures are optimized for medium-to-large objects. Since drones are typically very small and far away in our use case, we are injecting a **P2 (high-resolution) detection head** into the YOLOv11 configuration. This extracts features at a shallower layer (higher spatial resolution), giving the model significantly better sensitivity to small, fine-grained objects like distant drones. We generate this custom YAML file here before proceeding to the training step.

In [ ]:
import os

# Delete the old cached file
if os.path.exists('yolo11s-p2.yaml'):
    os.remove('yolo11s-p2.yaml')
    print("🗑️  Old file deleted")

# Write the corrected YAML
yaml_content = """
nc: 1

backbone:
  - [-1, 1, Conv, [32, 3, 2]]
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, C3k2, [128, False, 0.25]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 1, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 1, C3k2, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, C3k2, [512, True]]
  - [-1, 1, SPPF, [512, 5]]
  - [-1, 1, C2PSA, [512]]

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 1, C3k2, [256, False]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 1, C3k2, [128, False]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 2], 1, Concat, [1]]
  - [-1, 1, C3k2, [64, False]]

  - [16, 1, Conv, [128, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 1, C3k2, [256, False]]

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 1, C3k2, [512, True]]

  - [[19, 16, 22, 25], 1, Detect, [nc]]
"""

with open('yolo11s-p2.yaml', 'w') as f:
    f.write(yaml_content)
print("✅ New file written")

# Confirm what's on disk
with open('yolo11s-p2.yaml', 'r') as f:
    print(f.read())

### 4.2 Train YOLO

With our custom P2-head configuration (`yolo11s-p2.yaml`) generated in the previous step, we can now initialize the model and start training. This cell uses the custom YAML layout but loads the pre-trained `yolo11s.pt` weights for the standard layers to accelerate learning. We also apply drone-specific hyperparameter augmentations tailored for detecting small, distant objects.

In [ ]:
from ultralytics import YOLO
import time

# ── Configuration ───────────────────────────────────────────────
MODEL    = 'yolo11s-p2.yaml'
EPOCHS   = 150
IMGSZ    = 800
PROJECT  = 'drone_runs'
RUN_NAME = 'yolo11s_p2_v1'

BATCH    = 32

# ── Load pre-trained model ──────────────────────────────────────
model = YOLO(MODEL).load('yolo11s.pt')

# ── Start timer ─────────────────────────────────────────────────
train_start = time.time()

# ── Train ───────────────────────────────────────────────────────
results = model.train(
    data      = DATASET_YAML,
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    project   = PROJECT,
    name      = RUN_NAME,
    exist_ok  = True,

    # ── Optimizer ───────────────────────────────────────────────
    optimizer    = 'AdamW',
    lr0          = 0.001,
    lrf          = 0.01,
    warmup_epochs= 3,
    weight_decay = 0.0005,

    # ── Regularization ──────────────────────────────────────────
    # dropout=0.1 is only applied inside the classification head
    # in YOLO11; fine to keep for a small single-class dataset.
    dropout = 0.1,

    # ── Augmentation ────────────────────────────────────────────
    hsv_h    = 0.015,
    hsv_s    = 0.7,
    hsv_v    = 0.4,    # lowered from 0.5 — extreme brightness shifts
                       # can make small drones invisible; 0.4 is safer

    flipud   = 0.0,    # correct — drones don't fly inverted
    fliplr   = 0.5,
    mosaic   = 1.0,    # keep on for the first ~120 epochs (close_mosaic handles this)
    mixup    = 0.1,    # lowered from 0.15 — mixup hurts single-class
                       # localization; lighter value is better here
    degrees  = 15.0,   # raised from 10 — drones are seen at various
                       # angles from a fixed stereo rig; more rotation helps
    translate= 0.1,
    scale    = 0.6,    # raised from 0.5 — drones at 80 m (MAX_Z_M) are
                       # tiny; aggressive scale-down better mimics real range

    # close_mosaic: disable mosaic for the final N epochs so the model
    # fine-tunes on clean single images. Default is 10; keep it.
    close_mosaic = 10,

    # copy_paste: paste extra drone instances onto backgrounds.
    # Very effective for small single-class aerial detectors.
    copy_paste = 0.3,

    # ── Confidence alignment ────────────────────────────────────
    # Your tracker uses CONF_THRESH=0.45. Training with a matching
    # conf_thres during val ensures mAP is evaluated at the same
    # operating point you'll use at inference.
    conf = 0.45,       # val confidence threshold — matches CONF_THRESH

    # ── Early stopping ──────────────────────────────────────────
    patience  = 40,    # raised from 30 — with 150 epochs and AdamW,
                       # mAP can plateau for 30+ epochs before a late
                       # jump; 40 avoids premature termination

    # ── Hardware ────────────────────────────────────────────────
    device   = 0,
    workers  = 8,      # raised from 4 — A100 nodes typically have 8–16
                       # CPU cores; 8 workers keeps the data pipeline
                       # ahead of the GPU without thrashing
    amp      = True,   # mixed precision (BF16 on A100) — ~2× throughput,
                       # no accuracy cost on A100/A30 class hardware

    # ── Logging / saving ────────────────────────────────────────
    plots   = True,
    save    = True,
    verbose = True,

    # Save a checkpoint every 25 epochs so a Colab disconnect
    # doesn't lose everything.
    save_period = 25,
)

# ── Stop timer ──────────────────────────────────────────────────
train_end     = time.time()
training_time = train_end - train_start

hours   = int(training_time // 3600)
minutes = int((training_time % 3600) // 60)
seconds = int(training_time % 60)

print("\n✅ Training complete!")
print(f"⏱  Training time: {hours}h {minutes}m {seconds}s")
print(f"Best model saved at: {PROJECT}/{RUN_NAME}/weights/best.pt")

## 📊 Step 5 — Evaluate & Interpret Results

### 5.1 Training Curves and Confusion Matrix

First, we review the training loss curves and the confusion matrix. These visualizations help us understand if the model converged properly, whether it is overfitting, and how accurately it distinguishes the target classes.

In [ ]:
from IPython.display import Image, display
import os

results_dir = f"/content/runs/detect/{PROJECT}/{RUN_NAME}"

print("=" * 50)
print("TRAINING CURVES")
print("=" * 50)
display(Image(filename=f"{results_dir}/results.png"))

print("\nCONFUSION MATRIX")
display(Image(filename=f"{results_dir}/confusion_matrix_normalized.png"))

### 5.2 Validation Metrics

Next, we formally evaluate the best saved model on our validation set. This provides standard object detection metrics such as Mean Average Precision (mAP), Precision, and Recall to quantify the model's overall performance against our target thresholds.

In [ ]:
# ── Run validation on test set ──────────────────────────────────
best_model_path = f"/content/runs/detect/{PROJECT}/{RUN_NAME}/weights/best.pt"
model = YOLO(best_model_path)

# Validate on validation split
print("Validating on val split...")
val_metrics = model.val(
    data   = DATASET_YAML,
    split  = 'val',
    imgsz  = 1280,
    conf   = 0.25,
    iou    = 0.5,
    device = 0,
)

print("\n" + "=" * 50)
print("FINAL METRICS SUMMARY")
print("=" * 50)
print(f"  mAP50:       {val_metrics.box.map50:.4f}   (target: > 0.80)")
print(f"  mAP50-95:    {val_metrics.box.map:.4f}")
print(f"  Precision:   {val_metrics.box.mp:.4f}")
print(f"  Recall:      {val_metrics.box.mr:.4f}")

if val_metrics.box.map50 >= 0.80:
    print("\n  ✅ Target mAP50 > 0.80 achieved!")
else:
    print("\n  ⚠️  mAP50 below target — see tips in Step 6")

### 5.3 Visualizing Model Predictions

Finally, we run inference on a few random images from the validation split. Visualizing the predicted bounding boxes and confidence scores gives us a practical, real-world sense of how well the model detects small objects like drones.

In [ ]:
# ── Visualize predictions on validation images ──────────────────
import glob

val_img_dir = str(dataset_root / 'valid' / 'images')
if not os.path.exists(val_img_dir):
    val_img_dir = str(dataset_root / 'val' / 'images')

val_images = glob.glob(val_img_dir + '/*.jpg') + glob.glob(val_img_dir + '/*.png')
sample_imgs = random.sample(val_images, min(6, len(val_images)))

# Run inference
preds = model.predict(
    source = sample_imgs,
    conf   = 0.25,
    iou    = 0.5,
    imgsz  = 640,
    save   = False,
)

# Display results
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Model Predictions on Validation Set', fontsize=14, fontweight='bold')

for ax, result in zip(axes.flat, preds):
    img = result.plot()  # draws boxes on image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    n_boxes = len(result.boxes)
    ax.set_title(f'{n_boxes} detection(s)', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.savefig('val_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## ⬇️ Step 6 — Download Results
Download `best.pt` in additon to comprehensive summary of the training



In [ ]:
# ── Download model files to your local machine ──────────────────
from google.colab import files
import shutil
import torch
import os

# ── Load training arguments from checkpoint ─────────────────────
ckpt       = torch.load(best_model_path, map_location='cpu', weights_only=False)
train_args = ckpt.get('train_args', {})

# ── Package everything into a zip ───────────────────────────────
output_dir = 'model_output'
os.makedirs(output_dir, exist_ok=True)

shutil.copy(best_model_path,                    f'{output_dir}/best.pt')
shutil.copy(f"{results_dir}/results.png",       f'{output_dir}/training_curves.png')
shutil.copy('val_predictions.png',              f'{output_dir}/val_predictions.png')
shutil.copy('sample_annotations.png',           f'{output_dir}/sample_annotations.png')

# ── Write metrics report ─────────────────────────────────────────
with open(f'{output_dir}/metrics.txt', 'w') as f:

    f.write("=" * 40 + "\n")
    f.write("DATASET\n")
    f.write("=" * 40 + "\n")
    for split, count in split_counts.items():
        f.write(f"  {split:18s}: {count} images\n")
    f.write(f"  {'total':18s}: {total_images} images\n")
    f.write("\n")

    f.write("=" * 40 + "\n")
    f.write("TRAINING CONFIGURATION\n")
    f.write("=" * 40 + "\n")
    f.write(f"  {'Model':18s}: {MODEL}\n")
    f.write(f"  {'Epochs trained':18s}: {train_args.get('epochs', EPOCHS)}\n")
    f.write(f"  {'Image size':18s}: {train_args.get('imgsz', IMGSZ)}x{train_args.get('imgsz', IMGSZ)}\n")
    f.write(f"  {'Batch size':18s}: {train_args.get('batch', BATCH)}\n")
    f.write(f"  {'Training time':18s}: {hours}h {minutes}m {seconds}s\n")
    f.write(f"  {'GPU':18s}: {torch.cuda.get_device_name(0)}\n")
    f.write("\n")

    f.write("=" * 40 + "\n")
    f.write("OPTIMIZER & LEARNING RATE\n")
    f.write("=" * 40 + "\n")
    f.write(f"  {'Optimizer':18s}: {train_args.get('optimizer', 'AdamW')}\n")
    f.write(f"  {'Initial LR (lr0)':18s}: {train_args.get('lr0', 0.001)}\n")
    f.write(f"  {'Final LR (lrf)':18s}: {train_args.get('lrf', 0.01)}\n")
    f.write(f"  {'Warmup epochs':18s}: {train_args.get('warmup_epochs', 3)}\n")
    f.write(f"  {'Weight decay':18s}: {train_args.get('weight_decay', 0.0005)}\n")
    f.write(f"  {'Dropout':18s}: {train_args.get('dropout', 0.0)}\n")
    f.write(f"  {'Momentum':18s}: {train_args.get('momentum', 0.937)}\n")
    f.write("\n")

    f.write("=" * 40 + "\n")
    f.write("AUGMENTATION\n")
    f.write("=" * 40 + "\n")
    f.write(f"  {'Mosaic':18s}: {train_args.get('mosaic', 1.0)}\n")
    f.write(f"  {'Mixup':18s}: {train_args.get('mixup', 0.1)}\n")
    f.write(f"  {'Flip LR':18s}: {train_args.get('fliplr', 0.5)}\n")
    f.write(f"  {'Flip UD':18s}: {train_args.get('flipud', 0.0)}\n")
    f.write(f"  {'Rotation':18s}: {train_args.get('degrees', 10.0)} degrees\n")
    f.write(f"  {'HSV Hue':18s}: {train_args.get('hsv_h', 0.015)}\n")
    f.write(f"  {'HSV Saturation':18s}: {train_args.get('hsv_s', 0.7)}\n")
    f.write(f"  {'HSV Brightness':18s}: {train_args.get('hsv_v', 0.4)}\n")
    f.write(f"  {'Scale':18s}: {train_args.get('scale', 0.5)}\n")
    f.write(f"  {'Translate':18s}: {train_args.get('translate', 0.1)}\n")
    f.write("\n")

    f.write("=" * 40 + "\n")
    f.write("LOSS WEIGHTS\n")
    f.write("=" * 40 + "\n")
    f.write(f"  {'Box loss':18s}: {train_args.get('box', 7.5)}\n")
    f.write(f"  {'Class loss':18s}: {train_args.get('cls', 0.5)}\n")
    f.write(f"  {'DFL loss':18s}: {train_args.get('dfl', 1.5)}\n")
    f.write("\n")

    f.write("=" * 40 + "\n")
    f.write("PERFORMANCE METRICS\n")
    f.write("=" * 40 + "\n")
    f.write(f"  {'mAP50':18s}: {val_metrics.box.map50:.4f}\n")
    f.write(f"  {'mAP50-95':18s}: {val_metrics.box.map:.4f}\n")
    f.write(f"  {'Precision':18s}: {val_metrics.box.mp:.4f}\n")
    f.write(f"  {'Recall':18s}: {val_metrics.box.mr:.4f}\n")
    f.write("\n")


# ── Zip and download ─────────────────────────────────────────────
shutil.make_archive('drone_model_output', 'zip', output_dir)

print("Downloading drone_model_output.zip...")
files.download('drone_model_output.zip')
print("✅ Done! Check your Downloads folder.")


## 🔬 Step 7 - Testing

### 7.1 Test on any image from the Internet!

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import requests
from PIL import Image as PILImage
import matplotlib.pyplot as plt

# ── Load your trained model ─────────────────────────────────────
model = YOLO(best_model_path)

# ── Paste any image URL here (Or the path to the image) ────────────────────────────────────
image_url = "/content/prototype_photo.jpg"

# ── Run inference ───────────────────────────────────────────────
results = model.predict(
    source  = image_url,   # YOLO accepts URLs directly
    conf    = 0.05,        # your tuned threshold from Step 6
    iou     = 0.5,
    imgsz   = 640,
    save    = False,
)

# ── Display result ──────────────────────────────────────────────
import cv2
import numpy as np

result = results[0]
img = result.plot()   # draws boxes, labels, confidence scores
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Detections: {len(result.boxes)}", fontsize=14)
plt.show()

# ── Print detection details ─────────────────────────────────────
print(f"\nFound {len(result.boxes)} detection(s):\n")
for i, box in enumerate(result.boxes):
    cls_id = int(box.cls)
    conf   = float(box.conf)
    xyxy   = box.xyxy[0].tolist()  # [x1, y1, x2, y2] pixel coordinates
    print(f"  Detection {i+1}:")
    print(f"    Class:      {result.names[cls_id]}")
    print(f"    Confidence: {conf:.3f}")
    print(f"    Box:        x1={xyxy[0]:.0f}, y1={xyxy[1]:.0f}, x2={xyxy[2]:.0f}, y2={xyxy[3]:.0f}")

### 7.2 Test on a dataset
Upload a dataset to the session first then run the following. Make sure to adjust the path to the new dataset.

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
import os
import glob

# ── 1. Define the path to your NEW dataset's YAML file ──────────────
NEW_DATASET_YAML = DATASET_YAML  # ← CHANGE THIS PATH IF NEEDED

# ── 2. Load your trained 'best' model ────────────────────
model = YOLO(best_model_path)

# ── 3. Run Validation on the NEW dataset ────────────────────────────
print(f"Evaluating model on unseen dataset: {NEW_DATASET_YAML}...")
new_val_metrics = model.val(
    data   = NEW_DATASET_YAML,
    split  = 'test',
    imgsz  = 640,
    conf   = 0.25,
    iou    = 0.5,
    device = 0,
)

# ── 4. Print Results to Compare ─────────────────────────────────────
print("\n" + "=" * 50)
print("UNSEEN DATASET METRICS SUMMARY")
print("=" * 50)
print(f"  mAP50:       {new_val_metrics.box.map50:.4f}")
print(f"  mAP50-95:    {new_val_metrics.box.map:.4f}")
print(f"  Precision:   {new_val_metrics.box.mp:.4f}")
print(f"  Recall:      {new_val_metrics.box.mr:.4f}")

# ── 5. Display Evaluation Curves & Batched Images ───────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

print("\n" + "=" * 50)
print("VISUALIZING EVALUATION RESULTS")
print("=" * 50)

val_save_dir = new_val_metrics.save_dir

def show_images_in_grid(filenames, title, columns=2, figsize=(15, 6)):
    """Helper function to display a list of images in a neat grid."""
    # Filter out files that don't exist just in case
    existing_files = [f for f in filenames if os.path.exists(os.path.join(val_save_dir, f))]

    if not existing_files:
        return

    print(f"\n{title}")
    rows = (len(existing_files) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=figsize)

    # Flatten axes if it's a 2D array, or make it iterable if it's a single axis
    if rows > 1 or columns > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < len(existing_files):
            img_path = os.path.join(val_save_dir, existing_files[i])
            img = mpimg.imread(img_path)
            ax.imshow(img)
            ax.set_title(existing_files[i], fontsize=12, fontweight='bold')
        ax.axis('off') # Hide the axis ticks and lines

    plt.tight_layout()
    plt.show()

# 1. Display the 4 Curves (2 per row)
curve_files = [
    "BoxF1_curve.png",
    "BoxPR_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png"
]
show_images_in_grid(curve_files, "📈 Model Metric Curves:", columns=2, figsize=(16, 10))

# 2. Display the Confusion Matrices side-by-side (2 per row)
cm_files = [
    "confusion_matrix.png",
    "confusion_matrix_normalized.png"
]
show_images_in_grid(cm_files, "📊 Confusion Matrices:", columns=2, figsize=(16, 6))

# 3. Display Batch Predictions (Compare Ground Truth vs Predictions side-by-side)
print("\n🖼️ Sample Batch Predictions (Ground Truth vs Predictions):")
for batch_num in range(3): # For batches 0, 1, and 2
    batch_files = [
        f"val_batch{batch_num}_labels.jpg",
        f"val_batch{batch_num}_pred.jpg"
    ]
    # Check if this batch exists before trying to plot it
    if os.path.exists(os.path.join(val_save_dir, batch_files[0])):
        show_images_in_grid(batch_files, f"Batch {batch_num} Comparison:", columns=2, figsize=(20, 8))

### 7.3 Export Testing Metrics

In [ ]:
### Export testing metrics

import os
import glob
import shutil
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from google.colab import files

# ── 6. Save Metrics, Stitch Images, Clean Up, Zip, and Download ────────────

print(f"Processing results in: {val_save_dir}...")

# ==========================================================
# PART 1: Create the metrics summary text file
# ==========================================================
metrics_file_path = os.path.join(val_save_dir, "metrics_summary.txt")

with open(metrics_file_path, "w") as f:
    f.write("==================================================\n")
    f.write("UNSEEN DATASET METRICS SUMMARY\n")
    f.write("==================================================\n")
    f.write(f"  mAP50:       {new_val_metrics.box.map50:.4f}\n")
    f.write(f"  mAP50-95:    {new_val_metrics.box.map:.4f}\n")
    f.write(f"  Precision:   {new_val_metrics.box.mp:.4f}\n")
    f.write(f"  Recall:      {new_val_metrics.box.mr:.4f}\n")

print(f"📄 Saved metrics summary to: {metrics_file_path}")

# ==========================================================
# PART 2: Stitch the images together
# ==========================================================
def combine_and_save_images(image_filenames, output_filename, title, columns=2, figsize=(20, 12)):
    """Helper function to combine multiple images into one high-res image and save it."""
    existing_files = [os.path.join(val_save_dir, f) for f in image_filenames if os.path.exists(os.path.join(val_save_dir, f))]

    if not existing_files:
        return None

    rows = (len(existing_files) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=figsize)
    fig.suptitle(title, fontsize=24, fontweight='bold')

    if rows > 1 or columns > 1:
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, ax in enumerate(axes):
        if i < len(existing_files):
            img = mpimg.imread(existing_files[i])
            ax.imshow(img)

            # Clean up the subtitle for the batch images
            subtitle = os.path.basename(existing_files[i])
            if "labels" in subtitle: subtitle = "Ground Truth (Actual)"
            elif "pred" in subtitle: subtitle = "Model Predictions"

            ax.set_title(subtitle, fontsize=18)
        ax.axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Save the combined figure
    output_path = os.path.join(val_save_dir, output_filename)
    plt.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)

    return output_path

# A. Stitch Performance Curves
curve_files = ["BoxF1_curve.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png"]
combine_and_save_images(curve_files, "Combined_Metrics_Curves.png", "Model Performance Curves", columns=2, figsize=(20, 14))

# B. Stitch Confusion Matrices
cm_files = ["confusion_matrix.png", "confusion_matrix_normalized.png"]
combine_and_save_images(cm_files, "Combined_Confusion_Matrices.png", "Confusion Matrices", columns=2, figsize=(22, 10))

# C. Stitch Batch Images (Labels vs Predictions)
batch_label_files = glob.glob(os.path.join(val_save_dir, "val_batch*_labels.jpg"))
batch_originals_to_delete = []

for label_path in batch_label_files:
    # Extract just the base name (e.g., "val_batch0")
    base_name = os.path.basename(label_path).replace("_labels.jpg", "")
    pred_file = f"{base_name}_pred.jpg"
    label_file = f"{base_name}_labels.jpg"

    if os.path.exists(os.path.join(val_save_dir, pred_file)):
        combine_and_save_images(
            [label_file, pred_file],
            f"Combined_{base_name}.png",
            f"{base_name.replace('val_', '').capitalize()} Comparison",
            columns=2,
            figsize=(24, 12)
        )
        # Mark the originals for deletion
        batch_originals_to_delete.extend([label_file, pred_file])

print("🖼️ Successfully stitched curves, matrices, and batch images.")

# ==========================================================
# PART 3: Clean up the individual files!
# ==========================================================
print("🧹 Cleaning up individual original images...")

# Combine all the files we no longer need into one list
files_to_delete = curve_files + cm_files + batch_originals_to_delete

for filename in files_to_delete:
    file_path = os.path.join(val_save_dir, filename)
    if os.path.exists(file_path):
        os.remove(file_path)

# ==========================================================
# PART 4: Zip everything and download
# ==========================================================
zip_filename = "unseen_dataset_evaluation_results"

print(f"📦 Zipping cleaned results from: {val_save_dir}...")

# Zips the folder (now containing just the txt file and combined dashboards!)
shutil.make_archive(zip_filename, 'zip', val_save_dir)

print(f"⬇️ Downloading {zip_filename}.zip...")
files.download(f"{zip_filename}.zip")

print("✅ Done! Your downloaded zip is now perfectly clean and organized.")